In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import cv2
import matplotlib.pyplot as plt

In [ ]:
import comet_ml
comet_ml.login(project_name="comet-example-yolo26-coco128")

In [ ]:
from ultralytics import YOLO

# model = YOLO("yolov8n.pt")

In [ ]:
model = YOLO("yolov8n.pt")
# model = YOLO("yolov8m-pose.pt")
results = model.train(
    # data="diagdataset-pose.yaml",
    data="diagdataset.yaml",
    imgsz=640,
    batch=4,
    workers=2,
    epochs=20,
    project="y8-1"
)

In [ ]:
from src.utils.utils import imap

model = YOLO("y8-1/train4/weights/best.pt")

def visu(path):
    res = model(path, iou=0.1)
    for i in res:
        img = i.orig_img
        pred = imap(i.boxes.cls.cpu())
        bbs = i.boxes.cpu().xyxy
        # for i in i.keypoints.xy.cpu():
        #     p1, p2 = imap(i[0]), imap(i[1])
        #     cv2.line(img, p1, p2, (0, 255, 0), 2)
        for c, b in zip(pred, bbs):
            x1, y1, x2, y2 = imap(b)
            if c in {0, 2}:
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        plt.figure(figsize=(10, 10))
        plt.imshow(img)

visu("/home/petr/Desktop/diploma/example/Схемы/*")
# visu("/home/petr/study/diploma/src/visual/workspace/yolo/diagdataset/images/train/0a023e2c6ef143bb977520d208699c26.png")

In [ ]:
from utils.geom import bbox_of_line
from visual.models.diagram import DiagramDescription

img = cv2.imread("/home/petr/study/diploma/dataset/image/0a0f723303a54e47a2cb0523072b88d5.png")
dat = DiagramDescription.model_validate_json(open("/home/petr/study/diploma/dataset/label/0a0f723303a54e47a2cb0523072b88d5.json").read())

ims = img.shape[:2]

for i in dat.edges:
    b = bbox_of_line(i.points).int()
    # print(b)
    cv2.rectangle(img, b.p1, b.p2, (255, 0,0), 2)

plt.imshow(img)